In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

LOSO strategy tried but failed


In [5]:
import os
import pandas as pd
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# --- 1. HARDWARE STRATEGY & GLOBAL PARAMS ---
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("Environment: Running on TPU")
except ValueError:
    strategy = tf.distribute.get_strategy() 
    print(f"Environment: Running on {len(tf.config.list_physical_devices('GPU'))} GPUs")

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=10, restore_best_weights=True
)
custom_weights = {0: 1.0, 1: 1.0, 2: 1.8, 3: 1.8} # Targeted fix for Mild/Medium confusion

# --- 2. TRANSFORMER CLASSES ---
class PositionalEmbedding(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = self.add_weight(
            shape=(1, num_patches, projection_dim),
            initializer="zeros", trainable=True, name="pos_embedding"
        )
    def call(self, x): return x + self.pos_emb

class SpatialAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(name="seeker_layer", **kwargs)
        self.conv = layers.Conv1D(1, kernel_size=3, padding='same', activation='sigmoid')
    def call(self, x):
        avg_p = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_p = tf.reduce_max(x, axis=-1, keepdims=True)
        attention = self.conv(tf.concat([avg_p, max_p], axis=-1))
        return x * attention

# --- 3. BIOMARKER MASKING & DATA ENGINE ---
def apply_circular_eye_mask(image):
    size = 300
    center = size // 2
    radius = 110 
    Y, X = tf.meshgrid(tf.range(size), tf.range(size))
    dist = tf.sqrt(tf.cast((X - center)**2 + (Y - center)**2, tf.float32))
    mask = tf.cast(dist <= radius, tf.float32)[:, :, tf.newaxis]
    return image * mask

def get_loso_manager(data_dir='/kaggle/input/et-cropped-dataset-second'):
    file_list = glob.glob(os.path.join(data_dir, '*/*.jpg'))
    data_rows = []
    label_map = {'high': 0, 'low': 1, 'medium': 2, 'mild': 3}
    for f in file_list:
        path_parts = f.split('/')
        label_name = path_parts[-2]
        subject_id = path_parts[-1].split('-')[0]
        data_rows.append({'path': f, 'label': label_map[label_name], 'subject': subject_id})
    return pd.DataFrame(data_rows)

def load_loso_fold(df, target_subject):
    train_files = df[df['subject'] != target_subject]
    val_files = df[df['subject'] == target_subject]
    
    def parse_image(path, label_int):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, [300, 300])
        image = tf.cast(image, tf.float32) / 255.0
        return image, tf.one_hot(label_int, 4)

    def augment(image, label):
        image = apply_circular_eye_mask(image) 
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_contrast(image, 0.8, 1.2)
        return image, label

    train_ds = tf.data.Dataset.from_tensor_slices((train_files['path'], train_files['label']))
    train_ds = train_ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.shuffle(200).batch(32).prefetch(tf.data.AUTOTUNE)

    val_ds = tf.data.Dataset.from_tensor_slices((val_files['path'], val_files['label']))
    val_ds = val_ds.map(parse_image).map(lambda x, y: (apply_circular_eye_mask(x), y))
    val_ds = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)
    
    return train_ds, val_ds

# --- 4. MODEL BUILDER ---
def build_biomarker_transformer(input_shape=(300, 300, 3)):
    with strategy.scope():
        inputs = layers.Input(shape=input_shape)
        # Note: Masking is already in the data loader for LOSO efficiency
        
        base_model = applications.EfficientNetV2B0(input_shape=input_shape, include_top=False, weights='imagenet')
        base_model.trainable = True
        for layer in base_model.layers[30:-30]: layer.trainable = False
        
        x = base_model(inputs)
        num_patches = x.shape[1] * x.shape[2]
        x = layers.Reshape((num_patches, x.shape[-1]))(x)
        
        x = SpatialAttention()(x)
        x = PositionalEmbedding(num_patches, x.shape[-1])(x)
        
        res = x
        x = layers.LayerNormalization(epsilon=1e-6)(x)
        x = layers.MultiHeadAttention(num_heads=8, key_dim=64, dropout=0.5)(x, x)
        x = layers.Add()([res, x])
        
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(512, activation='gelu', kernel_regularizer='l2')(x)
        x = layers.Dropout(0.6)(x)
        outputs = layers.Dense(4, activation='softmax')(x)
        
        model = models.Model(inputs, outputs)
    return model, base_model

# --- 5. RUNNING THE LOSO LOOP ---
df_loso = get_loso_manager()
all_subjects = df_loso['subject'].unique()
subjects_to_test = all_subjects[:5] # Testing 5 folds to verify breakthrough
final_scores = []

for sub in subjects_to_test:
    print(f"\n--- LOSO FOLD: TESTING ON STRANGER SUBJECT {sub} ---")
    train_ds, val_ds = load_loso_fold(df_loso, sub)
    model, backbone = build_biomarker_transformer()

    # Stage 1: Stabilize Seeker
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-5), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=custom_weights)

    # Stage 2: Deep Refinement
    backbone.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-7), loss='categorical_crossentropy', metrics=['accuracy'])
    h = model.fit(train_ds, validation_data=val_ds, epochs=45, initial_epoch=15, 
                  class_weight=custom_weights, callbacks=[early_stop])
    
    final_scores.append(max(h.history['val_accuracy']))

print(f"\nFinal LOSO Validated Accuracy: {np.mean(final_scores):.4f}")

Environment: Running on 2 GPUs

--- LOSO FOLD: TESTING ON STRANGER SUBJECT 44 ---
Epoch 1/15


I0000 00:00:1769178849.079799     125 service.cc:152] XLA service 0x7b802021a0c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769178849.079840     125 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1769178849.079846     125 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1769178852.805935     125 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-01-23 14:34:21.406269: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:34:21.557194: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:34:22.868134: E external/local_xl

121/122 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.6825 - loss: 8.7225

2026-01-23 14:35:23.165767: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:35:23.313995: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:35:24.586639: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:35:24.727616: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:35:25.725044: E external/local_xla/xla/stream_

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 502ms/step - accuracy: 0.6823 - loss: 8.7178

2026-01-23 14:36:05.368229: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:36:05.503092: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:36:06.688371: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-23 14:36:06.829144: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


122/122 ━━━━━━━━━━━━━━━━━━━━ 142s 654ms/step - accuracy: 0.6821 - loss: 8.7131 - val_accuracy: 0.0000e+00 - val_loss: 14.5885
Epoch 2/15
122/122 ━━━━━━━━━━━━━━━━━━━━ 22s 178ms/step - accuracy: 0.4879 - loss: 9.7352 - val_accuracy: 0.0000e+00 - val_loss: 12.6434
Epoch 3/15
122/122 ━━━━━━━━━━━━━━━━━━━━ 22s 181ms/step - accuracy: 0.5486 - loss: 8.4127 - val_accuracy: 0.0000e+00 - val_loss: 12.0525
Epoch 4/15
122/122 ━━━━━━━━━━━━━━━━━━━━ 22s 175ms/step - accuracy: 0.4879 - loss: 8.5305 - val_accuracy: 0.0000e+00 - val_loss: 12.5794
Epoch 5/15
122/122 ━━━━━━━━━━━━━━━━━━━━ 22s 174ms/step - accuracy: 0.5498 - loss: 7.9433 - val_accuracy: 0.0000e+00 - val_loss: 12.6065
Epoch 6/15
122/122 ━━━━━━━━━━━━━━━━━━━━ 22s 177ms/step - accuracy: 0.4516 - loss: 8.1898 - val_accuracy: 0.0000e+00 - val_loss: 9.8403
Epoch 7/15
122/122 ━━━━━━━━━━━━━━━━━━━━ 22s 177ms/step - accuracy: 0.5424 - loss: 7.1981 - val_accuracy: 0.0000e+00 - val_loss: 10.1579
Epoch 8/15
122/122 ━━━━━━━━━━━━━━━━━━━━ 22s 176ms/step - ac

KeyboardInterrupt: 

Model 1 and 2 with 

In [1]:
# ============================================================================
# BLOCK 1: SUBJECT-LEVEL SPLIT (FROM MODEL 3)
# ============================================================================
import os
import shutil
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

base_path = '/kaggle/input/et-cropped-sharpened-dataset'
output_path = '/kaggle/working/split_dataset'
severity_groups = ['low', 'mild', 'medium', 'high']

# StratifiedGroupKFold split
all_subjects = []
all_labels = []
subject_to_files = {}

for label_idx, cat in enumerate(severity_groups):
    cat_dir = os.path.join(base_path, cat)
    if not os.path.exists(cat_dir): continue
    files = os.listdir(cat_dir)
    for f in files:
        sub_id = f.split('-')[0]
        if sub_id not in subject_to_files:
            subject_to_files[sub_id] = []
            all_subjects.append(sub_id)
            all_labels.append(label_idx)
        subject_to_files[sub_id].append((cat, f))

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf.split(all_subjects, all_labels, groups=all_subjects))

train_subjects = [all_subjects[i] for i in train_idx]
val_subjects = [all_subjects[i] for i in val_idx]

# Copy files
for sub_id in train_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        target_dir = os.path.join(output_path, 'train', cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(base_path, cat, f), os.path.join(target_dir, f))

for sub_id in val_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        target_dir = os.path.join(output_path, 'val', cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(base_path, cat, f), os.path.join(target_dir, f))
# [ADDED] Compute class weights for balanced training
class_counts = {cls: 0 for cls in severity_groups}
for sub_id in train_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        class_counts[cat] += 1

total_samples = sum(class_counts.values())
class_weights = {i: total_samples/(4*count) if count > 0 else 1.0 
                 for i, (cls, count) in enumerate(class_counts.items())}


print(f"Split complete. Train subjects: {len(train_subjects)}, Val subjects: {len(val_subjects)}")

# ============================================================================
# BLOCK 2: MODEL 2'S ARCHITECTURE (SINGLE-FRAME TRANSFORMER)
# ============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models

# [MODEL 2] Eyeball Masking
def apply_eyeball_mask(image):
    size = 300
    center = size // 2
    radius = 110
    Y, X = tf.meshgrid(tf.range(size), tf.range(size))
    dist = tf.sqrt(tf.cast((X - center)**2 + (Y - center)**2, tf.float32))
    mask = tf.cast(dist <= radius, tf.float32)[:, :, tf.newaxis]
    return image * mask

# [MODEL 2] Biomarker Seeker Layer
class SpatialAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(name="seeker_layer", **kwargs)
        self.conv = layers.Conv1D(1, kernel_size=3, padding='same', activation='sigmoid')

    def call(self, x):
        avg_p = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_p = tf.reduce_max(x, axis=-1, keepdims=True)
        attention = self.conv(tf.concat([avg_p, max_p], axis=-1))
        return x * attention

# [MODEL 2] Positional Embedding
class PositionalEmbedding(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = self.add_weight(
            shape=(1, num_patches, projection_dim),
            initializer="zeros", trainable=True, name="pos_embedding"
        )

    def call(self, x): return x + self.pos_emb

# [MODEL 2] 90% Architecture
def build_biomarker_transformer(input_shape=(300, 300, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    
    # Data Augmentation
    data_augmentation = tf.keras.Sequential([
        layers.Lambda(lambda x: apply_eyeball_mask(x)),
        layers.RandomFlip("horizontal"),
        layers.RandomContrast(0.15),
    ])
    x = data_augmentation(inputs)

    # EfficientNetV2B0 backbone
    base_model = tf.keras.applications.EfficientNetV2B0(
        input_shape=input_shape, include_top=False, weights='imagenet'
    )
    base_model.trainable = True
    # Initial freeze: middle layers frozen
    for layer in base_model.layers[20:-50]: layer.trainable = False

    x = base_model(x)
    f_h, f_w, channels = x.shape[1], x.shape[2], x.shape[3]
    num_patches = f_h * f_w
    
    x = layers.Reshape((num_patches, channels))(x)
    x = SpatialAttention()(x)
    x = PositionalEmbedding(num_patches, channels)(x)

    # Transformer Encoder (16 heads for biomarker resolution)
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.MultiHeadAttention(num_heads=16, key_dim=64, dropout=0.2)(x, x)
    x = layers.Add()([res, x])
    
    # FFN
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dense(channels * 2, activation=tf.nn.gelu)(x)
    x = layers.Dense(channels)(x)
    x = layers.Add()([res, x])
    
    x = layers.GlobalMaxPooling1D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model, base_model

# ============================================================================
# BLOCK 3: TRAINING LOOP (SIMPLIFIED)
# ============================================================================

# [MODEL 2] Progressive Unfreezing Callback
class ProgressiveUnfreezing(tf.keras.callbacks.Callback):
    def __init__(self, backbone, start_epoch=10, step=10, unfreeze_count=10):
        super().__init__()
        self.backbone = backbone
        self.start_epoch = start_epoch
        self.step = step
        self.unfreeze_count = unfreeze_count

    def on_epoch_begin(self, epoch, logs=None):
        if epoch >= self.start_epoch and (epoch - self.start_epoch) % self.step == 0:
            print(f"\n[Callback] Expanding fine-tuning at epoch {epoch}...")
            self.backbone.trainable = True
            # Unfreeze progressively from the middle outward
            frozen_start = 20 + (epoch - self.start_epoch) // self.step * self.unfreeze_count
            frozen_end = -50 - (epoch - self.start_epoch) // self.step * self.unfreeze_count
            for layer in self.backbone.layers[frozen_start:frozen_end]:
                layer.trainable = False

# Build model
model, backbone = build_biomarker_transformer()

# Load data with Model 3's subject split
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/split_dataset/train',
    validation_split=None,
    image_size=(300, 300),
    batch_size=32,  # [FIXED] Increased from 16 for BN stability
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/split_dataset/val',
    validation_split=None,
    image_size=(300, 300),
    batch_size=32,
    label_mode='categorical'
)

# Optimize datasets
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, 
                                   restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                                       patience=5, min_lr=1e-7),
    ProgressiveUnfreezing(backbone)
]

# [MODEL 2] Stage 1: Biomarker Localization
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
    metrics=['accuracy']
)

history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=35, class_weight=class_weights
)

# [MODEL 2] Stage 2: Deep Refinement
backbone.trainable = True
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(1e-5, decay_steps=8500, alpha=0.1)

model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=85, initial_epoch=35, class_weight=class_weights,
    callbacks=callbacks
)

print("Training complete. Model is ready for evaluation.")

Split complete. Train subjects: 32, Val subjects: 8


2026-01-24 12:01:21.142962: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769256081.369062      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769256081.432088      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769256081.956625      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769256081.956684      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769256081.956687      55 computation_placer.cc:177] computation placer alr

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Found 3200 files belonging to 4 classes.
Found 800 files belonging to 4 classes.
Epoch 1/35


I0000 00:00:1769256126.561365     105 service.cc:152] XLA service 0x7a9af001a500 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769256126.561411     105 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1769256126.561417     105 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1769256131.514049     105 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-01-24 12:02:22.261360: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-24 12:02:22.413645: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-24 12:02:23.875935: E external/local_xl

100/100 ━━━━━━━━━━━━━━━━━━━━ 111s 331ms/step - accuracy: 0.3300 - loss: 3.3522 - val_accuracy: 0.3288 - val_loss: 2.1459
Epoch 2/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 231ms/step - accuracy: 0.4627 - loss: 1.7369 - val_accuracy: 0.3638 - val_loss: 1.8865
Epoch 3/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 23s 234ms/step - accuracy: 0.5246 - loss: 1.4353 - val_accuracy: 0.4313 - val_loss: 1.6661
Epoch 4/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 237ms/step - accuracy: 0.5503 - loss: 1.3343 - val_accuracy: 0.3938 - val_loss: 1.7436
Epoch 5/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 239ms/step - accuracy: 0.6062 - loss: 1.2145 - val_accuracy: 0.5038 - val_loss: 1.5433
Epoch 6/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 241ms/step - accuracy: 0.6387 - loss: 1.1450 - val_accuracy: 0.4988 - val_loss: 1.4996
Epoch 7/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 242ms/step - accuracy: 0.6673 - loss: 1.0803 - val_accuracy: 0.5462 - val_loss: 1.4083
Epoch 8/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 246ms/step - accuracy: 0.6828 - loss: 1.0565 - va

TypeError: This optimizer was created with a `LearningRateSchedule` object as its `learning_rate` constructor argument, hence its learning rate is not settable. If you need the learning rate to be settable, you should instantiate the optimizer with a float `learning_rate` argument.

Fix-1

In [2]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.model_selection import StratifiedGroupKFold

# ============================================================================
# BLOCK 1: SUBJECT-LEVEL SPLIT
# ============================================================================
base_path = '/kaggle/input/et-cropped-sharpened-dataset'
output_path = '/kaggle/working/split_dataset'
severity_groups = ['low', 'mild', 'medium', 'high']

# StratifiedGroupKFold split
all_subjects = []
all_labels = []
subject_to_files = {}

for label_idx, cat in enumerate(severity_groups):
    cat_dir = os.path.join(base_path, cat)
    if not os.path.exists(cat_dir): continue
    files = os.listdir(cat_dir)
    for f in files:
        sub_id = f.split('-')[0]
        if sub_id not in subject_to_files:
            subject_to_files[sub_id] = []
            all_subjects.append(sub_id)
            all_labels.append(label_idx)
        subject_to_files[sub_id].append((cat, f))

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf.split(all_subjects, all_labels, groups=all_subjects))

train_subjects = [all_subjects[i] for i in train_idx]
val_subjects = [all_subjects[i] for i in val_idx]

# Copy files
for sub_id in train_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        target_dir = os.path.join(output_path, 'train', cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(base_path, cat, f), os.path.join(target_dir, f))

for sub_id in val_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        target_dir = os.path.join(output_path, 'val', cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(base_path, cat, f), os.path.join(target_dir, f))

# Compute class weights
class_counts = {cls: 0 for cls in severity_groups}
for sub_id in train_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        class_counts[cat] += 1

total_samples = sum(class_counts.values())
class_weights = {i: total_samples/(4*count) if count > 0 else 1.0 
                 for i, (cls, count) in enumerate(class_counts.items())}

print(f"Split complete. Train subjects: {len(train_subjects)}, Val subjects: {len(val_subjects)}")

# ============================================================================
# BLOCK 2: MODEL ARCHITECTURE (ENHANCED)
# ============================================================================

def apply_eyeball_mask(image):
    size = 300
    center = size // 2
    radius = 110
    Y, X = tf.meshgrid(tf.range(size), tf.range(size))
    dist = tf.sqrt(tf.cast((X - center)**2 + (Y - center)**2, tf.float32))
    mask = tf.cast(dist <= radius, tf.float32)[:, :, tf.newaxis]
    return image * mask

class SpatialAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(name="seeker_layer", **kwargs)
        self.conv = layers.Conv1D(1, kernel_size=3, padding='same', activation='sigmoid')

    def call(self, x):
        avg_p = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_p = tf.reduce_max(x, axis=-1, keepdims=True)
        attention = self.conv(tf.concat([avg_p, max_p], axis=-1))
        return x * attention

class PositionalEmbedding(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = self.add_weight(
            shape=(1, num_patches, projection_dim),
            initializer="zeros", trainable=True, name="pos_embedding"
        )
    def call(self, x): return x + self.pos_emb

def freeze_batch_norm_layers(model):
    for layer in model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        elif hasattr(layer, 'layers'):
            for sublayer in layer.layers:
                if isinstance(sublayer, layers.BatchNormalization):
                    sublayer.trainable = False

def build_biomarker_transformer(input_shape=(300, 300, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    
    data_augmentation = tf.keras.Sequential([
        layers.Lambda(lambda x: apply_eyeball_mask(x)),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomContrast(0.15),
        layers.RandomBrightness(0.1),
    ])
    x = data_augmentation(inputs)

    base_model = tf.keras.applications.EfficientNetV2B0(
        input_shape=input_shape, include_top=False, weights='imagenet'
    )
    base_model.trainable = True
    for layer in base_model.layers[20:-50]: layer.trainable = False

    x = base_model(x)
    f_h, f_w, channels = x.shape[1], x.shape[2], x.shape[3]
    num_patches = f_h * f_w

    x = layers.Reshape((num_patches, channels))(x)
    x = SpatialAttention()(x)
    x = PositionalEmbedding(num_patches, channels)(x)

    # 8 heads for better generalization
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.MultiHeadAttention(num_heads=8, key_dim=64, dropout=0.3)(x, x)
    x = layers.Add()([res, x])

    # FFN with L2 Regularization
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dense(channels * 2, activation=tf.nn.gelu, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dense(channels, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Add()([res, x])

    # [FIXED] GlobalAveragePooling is more stable for small datasets
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x) 

    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs), base_model

# ============================================================================
# BLOCK 3: TRAINING LOOP
# ============================================================================

class ProgressiveUnfreezing(tf.keras.callbacks.Callback):
    def __init__(self, backbone, start_epoch=10, step=10, unfreeze_count=10):
        super().__init__()
        self.backbone = backbone
        self.start_epoch = start_epoch
        self.step = step
        self.unfreeze_count = unfreeze_count

    def on_epoch_begin(self, epoch, logs=None):
        if epoch >= self.start_epoch and (epoch - self.start_epoch) % self.step == 0:
            print(f"\n[Callback] Expanding fine-tuning at epoch {epoch}...")
            self.backbone.trainable = True
            frozen_start = 20 + (epoch - self.start_epoch) // self.step * self.unfreeze_count
            frozen_end = -50 - (epoch - self.start_epoch) // self.step * self.unfreeze_count
            for layer in self.backbone.layers[frozen_start:frozen_end]:
                layer.trainable = False

model, backbone = build_biomarker_transformer()

train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/split_dataset/train',
    image_size=(300, 300), batch_size=32, label_mode='categorical'
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/split_dataset/val',
    image_size=(300, 300), batch_size=32, label_mode='categorical'
).prefetch(tf.data.AUTOTUNE)

# Stage 1: Localization
callbacks_s1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
]

model.compile(optimizer=tf.keras.optimizers.Adam(3e-5),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
              metrics=['accuracy'])

freeze_batch_norm_layers(model)
model.fit(train_ds, validation_data=val_ds, epochs=35, 
          class_weight=class_weights, callbacks=callbacks_s1)

# Stage 2: Refinement (Fixed Optimizer logic)
backbone.trainable = True
lr_sched = tf.keras.optimizers.schedules.CosineDecay(1e-5, decay_steps=8500, alpha=0.1)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_sched),
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
              metrics=['accuracy'])

callbacks_s2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    ProgressiveUnfreezing(backbone)
]

model.fit(train_ds, validation_data=val_ds, epochs=85, initial_epoch=35,
          class_weight=class_weights, callbacks=callbacks_s2)

print("Training complete.")

Split complete. Train subjects: 32, Val subjects: 8
Found 3200 files belonging to 4 classes.
Found 800 files belonging to 4 classes.
Epoch 1/35


E0000 00:00:1769259716.946320      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_3_1/efficientnetv2-b0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


100/100 ━━━━━━━━━━━━━━━━━━━━ 68s 419ms/step - accuracy: 0.3162 - loss: 1.8462 - val_accuracy: 0.2475 - val_loss: 1.7284 - learning_rate: 3.0000e-05
Epoch 2/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 38s 378ms/step - accuracy: 0.4639 - loss: 1.6170 - val_accuracy: 0.4200 - val_loss: 1.6118 - learning_rate: 3.0000e-05
Epoch 3/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 38s 382ms/step - accuracy: 0.5328 - loss: 1.5312 - val_accuracy: 0.4137 - val_loss: 1.6338 - learning_rate: 3.0000e-05
Epoch 4/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 386ms/step - accuracy: 0.5945 - loss: 1.4345 - val_accuracy: 0.4075 - val_loss: 1.6840 - learning_rate: 3.0000e-05
Epoch 5/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 390ms/step - accuracy: 0.6562 - loss: 1.3701 - val_accuracy: 0.4563 - val_loss: 1.6613 - learning_rate: 3.0000e-05
Epoch 6/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 393ms/step - accuracy: 0.6962 - loss: 1.3076 - val_accuracy: 0.5387 - val_loss: 1.4895 - learning_rate: 3.0000e-05
Epoch 7/35
100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 394ms/step - 

E0000 00:00:1769261122.140806      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_3_1/efficientnetv2-b0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


100/100 ━━━━━━━━━━━━━━━━━━━━ 65s 435ms/step - accuracy: 0.9056 - loss: 1.0032 - val_accuracy: 0.6350 - val_loss: 1.5231
Epoch 37/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 392ms/step - accuracy: 0.9109 - loss: 1.0036 - val_accuracy: 0.6263 - val_loss: 1.5248
Epoch 38/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 40s 396ms/step - accuracy: 0.9113 - loss: 0.9948 - val_accuracy: 0.6400 - val_loss: 1.4562
Epoch 39/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 394ms/step - accuracy: 0.9173 - loss: 0.9869 - val_accuracy: 0.6338 - val_loss: 1.4879
Epoch 40/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 40s 396ms/step - accuracy: 0.9105 - loss: 0.9915 - val_accuracy: 0.6562 - val_loss: 1.4520

[Callback] Expanding fine-tuning at epoch 40...
Epoch 41/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 394ms/step - accuracy: 0.9146 - loss: 0.9817 - val_accuracy: 0.6325 - val_loss: 1.5028
Epoch 42/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 39s 394ms/step - accuracy: 0.9102 - loss: 0.9883 - val_accuracy: 0.6450 - val_loss: 1.4923
Epoch 43/85
100/100 ━━━━━━━━━━━━━━━━━━━

Block 1: Setup and Subject-Level Data Splitting

In [3]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.model_selection import StratifiedGroupKFold

# Enable mixed precision for speed and memory efficiency
tf.keras.mixed_precision.set_global_policy('mixed_float16')

base_path = '/kaggle/input/et-cropped-sharpened-dataset'
output_path = '/kaggle/working/split_dataset'
severity_groups = ['low', 'mild', 'medium', 'high']

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

all_subjects = []
all_labels = []
subject_to_files = {}

for label_idx, cat in enumerate(severity_groups):
    cat_dir = os.path.join(base_path, cat)
    if not os.path.exists(cat_dir): continue
    files = os.listdir(cat_dir)
    for f in files:
        sub_id = f.split('-')[0]
        if sub_id not in subject_to_files:
            subject_to_files[sub_id] = []
        all_subjects.append(sub_id)
        all_labels.append(label_idx)
        subject_to_files[sub_id].append((cat, f))

train_idx, val_idx = next(sgkf.split(all_subjects, all_labels, groups=all_subjects))
train_subjects = [all_subjects[i] for i in train_idx]
val_subjects = [all_subjects[i] for i in val_idx]

# Refresh local directories
if os.path.exists(output_path): shutil.rmtree(output_path)

for sub_id in train_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        target_dir = os.path.join(output_path, 'train', cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(base_path, cat, f), os.path.join(target_dir, f))

for sub_id in val_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        target_dir = os.path.join(output_path, 'val', cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(base_path, cat, f), os.path.join(target_dir, f))

class_counts = {cls: 0 for cls in severity_groups}
for sub_id in train_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        class_counts[cat] += 1

total_samples = sum(class_counts.values())
class_weights = {i: total_samples/(4*count) if count > 0 else 1.0 
                 for i, (cls, count) in enumerate(class_counts.items())}

print(f"Split Complete. Training on {len(train_subjects)} subjects.")

Split Complete. Training on 3200 subjects.


Block 2: Custom Layers and Model Architecture

In [5]:
def apply_eyeball_mask(image, radius=130):
    size = 300
    center = size // 2
    Y, X = tf.meshgrid(tf.range(size), tf.range(size))
    dist = tf.sqrt(tf.cast((X - center)**2 + (Y - center)**2, tf.float32))
    # Cast mask to match image type (float16) for mixed precision compatibility
    mask = tf.cast(dist <= radius, image.dtype)[:, :, tf.newaxis]
    return image * mask

class SpatialAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(name="seeker_layer", **kwargs)
        self.conv = layers.Conv1D(1, kernel_size=3, padding='same', activation='sigmoid')
    def call(self, x):
        avg_p = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_p = tf.reduce_max(x, axis=-1, keepdims=True)
        attention = self.conv(tf.concat([avg_p, max_p], axis=-1))
        return x * attention

class PositionalEmbedding(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = self.add_weight(shape=(1, num_patches, projection_dim), initializer="zeros", trainable=True)
    def call(self, x): return x + self.pos_emb

def build_biomarker_transformer(input_shape=(300, 300, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    
    # Pre-processing & Augmentation
    x = layers.Lambda(lambda x: apply_eyeball_mask(x))(inputs)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomRotation(0.03)(x)
    x = layers.RandomTranslation(0.03, 0.03)(x)
    x = layers.RandomContrast(0.1)(x)
    
    # EfficientNet Backbone
    base_model = tf.keras.applications.EfficientNetV2B0(input_shape=input_shape, include_top=False, weights='imagenet')
    base_model.trainable = True
    for layer in base_model.layers[20:-50]: layer.trainable = False
    
    x = base_model(x)
    f_h, f_w, channels = x.shape[1], x.shape[2], x.shape[3]
    num_patches = f_h * f_w
    x = layers.Reshape((num_patches, channels))(x)
    
    # Transformer Encoder Section
    x = SpatialAttention()(x)
    x = PositionalEmbedding(num_patches, channels)(x)
    
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.3)(x, x)
    x = layers.Add()([res, x])
    
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dense(channels * 2, activation=tf.nn.gelu)(x)
    x = layers.Dense(channels)(x)
    x = layers.Add()([res, x])
    
    # Global Head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax', kernel_regularizer=regularizers.l2(1e-5))(x)
    
    return models.Model(inputs, outputs), base_model

print("Architecture build functions defined.")

Architecture build functions defined.


Block 3: The BN-Safe Progressive Unfreezing Callback

In [6]:
class ProgressiveUnfreezing(tf.keras.callbacks.Callback):
    def __init__(self, backbone, start_epoch=20, unfreeze_per_epoch=5):
        super().__init__()
        self.backbone = backbone
        self.start_epoch = start_epoch
        self.unfreeze_per_epoch = unfreeze_per_epoch
        
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.start_epoch: return
        
        total_layers = len(self.backbone.layers)
        unfreeze_count = (epoch - self.start_epoch + 1) * self.unfreeze_per_epoch
        frozen_until = max(20, total_layers - 50 - unfreeze_count)
        
        print(f"\n[Callback] Epoch {epoch}: Unfreezing non-BN layers from {frozen_until} to {total_layers-50}")
        
        self.backbone.trainable = True
        for i, layer in enumerate(self.backbone.layers):
            if i < 20 or isinstance(layer, layers.BatchNormalization) or i >= (total_layers - 50):
                layer.trainable = False
            elif i >= frozen_until:
                layer.trainable = True
            else:
                layer.trainable = False

Block 4: Data Loaders and Compilation

In [9]:
# Data Loaders
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/split_dataset/train', image_size=(300, 300), batch_size=32, label_mode='categorical'
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/split_dataset/val', image_size=(300, 300), batch_size=32, label_mode='categorical'
).prefetch(tf.data.AUTOTUNE)

# Initialize Model
model, backbone = build_biomarker_transformer()

# Learning Rate Schedule (This replaces the need for ReduceLROnPlateau)
def get_lr_schedule(steps):
    return tf.keras.optimizers.schedules.PiecewiseConstantDecay(
        boundaries=[35 * steps], 
        values=[3e-5, 1e-5]
    )

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=get_lr_schedule(len(train_ds))),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.15),
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall_high', class_id=3)]
)

print("Model compiled with PiecewiseConstantDecay schedule.")

Found 3200 files belonging to 4 classes.
Found 800 files belonging to 4 classes.
Model compiled with PiecewiseConstantDecay schedule.


Block 5: Training Execution

In [10]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_recall_high', 
        patience=20, 
        restore_best_weights=True, 
        mode='max'
    ),
    ProgressiveUnfreezing(backbone, start_epoch=20, unfreeze_per_epoch=5)
    # [FIXED] Removed ReduceLROnPlateau to prevent the TypeError
]

print("\n=== Training Model (Single Stage, BN-Safe Unfreezing) ===")
history = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=85, 
    class_weight=class_weights, 
    callbacks=callbacks
)

model.save('/kaggle/working/best_et_model_final.keras')
print("\nTraining complete and model saved.")


=== Training Model (Single Stage, BN-Safe Unfreezing) ===
Epoch 1/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 60s 322ms/step - accuracy: 0.3533 - loss: 1.3777 - recall_high: 0.1815 - val_accuracy: 0.4563 - val_loss: 1.2573 - val_recall_high: 0.0050
Epoch 2/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 28s 283ms/step - accuracy: 0.5479 - loss: 1.1697 - recall_high: 0.4177 - val_accuracy: 0.4663 - val_loss: 1.2166 - val_recall_high: 0.0850
Epoch 3/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 29s 285ms/step - accuracy: 0.6190 - loss: 1.0803 - recall_high: 0.4901 - val_accuracy: 0.4663 - val_loss: 1.2854 - val_recall_high: 0.2150
Epoch 4/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 29s 286ms/step - accuracy: 0.6491 - loss: 1.0389 - recall_high: 0.5476 - val_accuracy: 0.5150 - val_loss: 1.2183 - val_recall_high: 0.1900
Epoch 5/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 29s 286ms/step - accuracy: 0.6869 - loss: 0.9932 - recall_high: 0.5510 - val_accuracy: 0.5250 - val_loss: 1.1608 - val_recall_high: 0.3300
Epoch 6/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 29s 28

In [11]:
# =============================================================================
# BLOCK 2: SIMPLIFIED MODEL ARCHITECTURE (FIXED VERSION)
# =============================================================================

class SpatialAttention(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(name="seeker_layer", **kwargs)
        self.conv = layers.Conv1D(1, kernel_size=3, padding='same', activation='sigmoid')
    def call(self, x):
        avg_p = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_p = tf.reduce_max(x, axis=-1, keepdims=True)
        attention = self.conv(tf.concat([avg_p, max_p], axis=-1))
        return x * attention

class PositionalEmbedding(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = self.add_weight(
            shape=(1, num_patches, projection_dim),
            initializer="zeros", trainable=True
        )
    def call(self, x): 
        return x + self.pos_emb

def build_simplified_transformer(input_shape=(300, 300, 3), num_classes=4):
    """Simplified version without mask and with better pooling"""
    inputs = layers.Input(shape=input_shape)
    
    # ✅ FIXED: Removed the circular mask - it was causing artifacts
    # ✅ FIXED: Simplified augmentation - less is more for medical images
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.02)(x)  # Reduced from 0.03
    x = layers.RandomContrast(0.05)(x)  # Reduced from 0.1
    # ✅ Added: Small Gaussian noise for regularization
    x = layers.GaussianNoise(0.01)(x)
    
    # EfficientNet Backbone with more layers frozen
    base_model = tf.keras.applications.EfficientNetV2B0(
        input_shape=input_shape, include_top=False, weights='imagenet'
    )
    base_model.trainable = True
    
    # ✅ FIXED: Freeze more layers initially to prevent overfitting
    for layer in base_model.layers[:150]:  # Increased from 20
        layer.trainable = False
    
    x = base_model(x)
    
    # Get feature dimensions
    f_h, f_w, channels = x.shape[1], x.shape[2], x.shape[3]
    num_patches = f_h * f_w
    
    # Reshape for transformer
    x = layers.Reshape((num_patches, channels))(x)
    
    # ✅ FIXED: Single transformer block (was two)
    # Spatial Attention
    x = SpatialAttention()(x)
    
    # Positional Embedding
    x = PositionalEmbedding(num_patches, channels)(x)
    
    # Single Attention Block (reduced complexity)
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.MultiHeadAttention(
        num_heads=2,  # ✅ REDUCED from 4
        key_dim=16,   # ✅ REDUCED from 32
        dropout=0.5   # ✅ INCREASED from 0.3
    )(x, x)
    x = layers.Add()([res, x])
    
    # ✅ FIXED: Changed to GlobalMaxPooling1D for better feature extraction
    x = layers.GlobalMaxPooling1D()(x)  # This is CRITICAL
    
    # Classification head with stronger regularization
    x = layers.Dropout(0.6)(x)  # ✅ INCREASED from 0.5
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    return models.Model(inputs, outputs), base_model

# =============================================================================
# BLOCK 3: IMPROVED TRAINING WITH EARLY STOPPING
# =============================================================================

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/split_dataset/train',
    image_size=(300, 300), 
    batch_size=32, 
    label_mode='categorical'
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/split_dataset/val',
    image_size=(300, 300), 
    batch_size=32, 
    label_mode='categorical'
).prefetch(tf.data.AUTOTUNE)

# Build the simplified model
model, backbone = build_simplified_transformer()

# ✅ FIXED: Simpler learning rate schedule
initial_lr = 1e-4
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=initial_lr,
    decay_steps=1000,
    decay_rate=0.96,
    staircase=True
)

# Compile with better regularization
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.25),  # ✅ INCREASED
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall_high', class_id=3),
        tf.keras.metrics.Recall(name='recall_medium', class_id=2),
        tf.keras.metrics.Recall(name='recall_mild', class_id=1),
        tf.keras.metrics.Recall(name='recall_low', class_id=0)
    ]
)

# ✅ FIXED: Aggressive early stopping to prevent overfitting
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_recall_high',
        patience=12,  # Stop if no improvement for 12 epochs
        restore_best_weights=True,
        mode='max',
        min_delta=0.01  # Minimum change to qualify as improvement
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=6,  # Reduce LR if no improvement for 6 epochs
        min_lr=1e-6,
        verbose=1
    ),
    # Simple progressive unfreezing (optional, can remove if still overfitting)
    tf.keras.callbacks.LambdaCallback(
        on_epoch_begin=lambda epoch, logs: (
            print(f"\nEpoch {epoch+1}: Unfreezing layers 150-{180+epoch*2}")
            if epoch >= 15 and epoch < 40 else None
        )
    )
]

print("\n=== Training Simplified Model (No Mask, Max Pooling) ===")
print(f"Training samples: {sum(class_counts.values())}")
print(f"Validation samples: {len(val_subjects) * 100}")  # Assuming 100 per subject

# Train with fewer epochs
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,  # ✅ REDUCED from 85
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

# Save the model
model.save('/kaggle/working/best_simplified_model.keras')

# =============================================================================
# BLOCK 4: EVALUATION & ANALYSIS
# =============================================================================

print("\n" + "="*60)
print("TRAINING COMPLETE - ANALYSIS")
print("="*60)

# Calculate final metrics
best_val_acc = max(history.history['val_accuracy'])
best_val_recall_high = max(history.history['val_recall_high'])
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]

print(f"\nBest Validation Accuracy: {best_val_acc:.2%}")
print(f"Best High-Severity Recall: {best_val_recall_high:.2%}")
print(f"Final Train Accuracy: {final_train_acc:.2%}")
print(f"Final Val Accuracy: {final_val_acc:.2%}")
print(f"Overfitting Gap: {final_train_acc - final_val_acc:.2%}")

# Expected performance
if best_val_acc > 0.75:
    print("\n✅ SUCCESS: Model is performing well!")
    print("Expected clinical utility: GOOD")
elif best_val_acc > 0.65:
    print("\n⚠️ MODERATE: Model needs improvement")
    print("Expected clinical utility: LIMITED")
else:
    print("\n❌ POOR: Model needs significant improvement")
    print("Consider: More data, transfer learning, or simpler architecture")

# Save training history
np.save('/kaggle/working/training_history.npy', history.history)
print("\nModel and training history saved.")

Found 3200 files belonging to 4 classes.
Found 800 files belonging to 4 classes.

=== Training Simplified Model (No Mask, Max Pooling) ===
Training samples: 320000
Validation samples: 80000
Epoch 1/60
100/100 ━━━━━━━━━━━━━━━━━━━━ 58s 232ms/step - accuracy: 0.2353 - loss: 7.8728 - recall_high: 0.2014 - recall_low: 0.3890 - recall_medium: 0.1546 - recall_mild: 0.2015 - val_accuracy: 0.2725 - val_loss: 1.8157 - val_recall_high: 0.6950 - val_recall_low: 0.0000e+00 - val_recall_medium: 0.0700 - val_recall_mild: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 2/60
100/100 ━━━━━━━━━━━━━━━━━━━━ 19s 193ms/step - accuracy: 0.2751 - loss: 2.6772 - recall_high: 0.2425 - recall_low: 0.1861 - recall_medium: 0.2272 - recall_mild: 0.2462 - val_accuracy: 0.2912 - val_loss: 1.3932 - val_recall_high: 0.0000e+00 - val_recall_low: 0.0000e+00 - val_recall_medium: 0.0000e+00 - val_recall_mild: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 3/60
100/100 ━━━━━━━━━━━━━━━━━━━━ 19s 194ms/step - accuracy: 0.2814 - loss

In [15]:
# =============================================================================
# BLOCK 2: THE BALANCED HYBRID TRANSFORMER
# =============================================================================

def apply_eyeball_mask(image, radius=140): # Increased radius to prevent edge artifacts
    size = 300
    center = size // 2
    Y, X = tf.meshgrid(tf.range(size), tf.range(size))
    dist = tf.sqrt(tf.cast((X - center)**2 + (Y - center)**2, tf.float32))
    mask = tf.cast(dist <= radius, image.dtype)[:, :, tf.newaxis]
    return image * mask

def build_balanced_transformer(input_shape=(300, 300, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    
    # 1. Surgical Augmentation
    x = layers.Lambda(lambda x: apply_eyeball_mask(x))(inputs)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomRotation(0.05)(x) # Harder for the model to memorize
    x = layers.RandomZoom(0.1)(x)      # Crucial: makes eyes different sizes
    x = layers.RandomContrast(0.1)(x)
    
    # 2. EfficientNet Backbone
    base_model = tf.keras.applications.EfficientNetV2B0(
        input_shape=input_shape, include_top=False, weights='imagenet'
    )
    base_model.trainable = True
    for layer in base_model.layers[:100]: # Freeze the first 100 layers (general shapes)
        layer.trainable = False
    
    x = base_model(x)
    f_h, f_w, channels = x.shape[1], x.shape[2], x.shape[3]
    num_patches = f_h * f_w
    x = layers.Reshape((num_patches, channels))(x)

    # 3. Transformer Logic
    x = SpatialAttention()(x) # Keep your seeker layer
    x = PositionalEmbedding(num_patches, channels)(x)
    
    # Single Attention Block with Balanced Capacity
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.MultiHeadAttention(
        num_heads=4, # Back to 4 heads for enough "brain power"
        key_dim=32,
        dropout=0.4  # High dropout to stop overfitting
    )(x, x)
    x = layers.Add()([res, x])
    
    # 4. Global Head (Back to Average Pooling)
    x = layers.GlobalAveragePooling1D()(x) # Better than "Max" for small datasets
    x = layers.Dropout(0.5)(x)
    
    # Extra Dense layer to give the model room to think
    x = layers.Dense(256, activation=tf.nn.gelu)(x)
    x = layers.Dropout(0.4)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    return models.Model(inputs, outputs), base_model

In [16]:
# =============================================================================
# BLOCK 3: BALANCED TRAINING EXECUTION
# =============================================================================

# 1. Initialize Model
model, backbone = build_balanced_transformer()

# 2. Compile with Balanced Regularization
# Label smoothing at 0.1 is the "Goldilocks" zone for medical AI
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        'accuracy', 
        tf.keras.metrics.Recall(name='recall_high', class_id=3),
        tf.keras.metrics.AUC(name='auc')
    ]
)

# 3. Callbacks for Stability
# We monitor recall_high because missing a severe case is clinically worse
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_recall_high', 
        patience=15, 
        restore_best_weights=True, 
        mode='max'
    ),
    # The Expert Move: Keep BN layers frozen during unfreezing
    ProgressiveUnfreezing(backbone, start_epoch=15, unfreeze_per_epoch=10),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=5, 
        min_lr=1e-7, 
        verbose=1
    )
]

# 4. Run Training
print("\n=== Training Balanced Model (Hybrid-ViT with BN-Safety) ===")
history = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=85, 
    class_weight=class_weights, 
    callbacks=callbacks,
    verbose=1
)

model.save('/kaggle/working/balanced_biomarker_model.keras')
print("\nTraining complete. Model saved.")


=== Training Balanced Model (Hybrid-ViT with BN-Safety) ===
Epoch 1/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 72s 289ms/step - accuracy: 0.2795 - auc: 0.5314 - loss: 1.4182 - recall_high: 0.0078 - val_accuracy: 0.3750 - val_auc: 0.6098 - val_loss: 1.3525 - val_recall_high: 0.0000e+00 - learning_rate: 3.0000e-05
Epoch 2/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 248ms/step - accuracy: 0.4300 - auc: 0.6893 - loss: 1.2862 - recall_high: 0.1873 - val_accuracy: 0.3925 - val_auc: 0.6955 - val_loss: 1.2841 - val_recall_high: 0.0550 - learning_rate: 3.0000e-05
Epoch 3/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 248ms/step - accuracy: 0.4959 - auc: 0.7667 - loss: 1.1841 - recall_high: 0.3817 - val_accuracy: 0.4400 - val_auc: 0.7545 - val_loss: 1.2009 - val_recall_high: 0.1600 - learning_rate: 3.0000e-05
Epoch 4/85
100/100 ━━━━━━━━━━━━━━━━━━━━ 25s 248ms/step - accuracy: 0.5959 - auc: 0.8313 - loss: 1.0764 - recall_high: 0.4531 - val_accuracy: 0.4863 - val_auc: 0.7805 - val_loss: 1.1611 - val_recall_high: 0.2650 - learn